# Week 1, Friday: Explore the patent dataset

Input: `data/processed/families_tagged.csv` (made by `src/tag_patents.py`). Output: charts in `reports/figures/` and `reports/week1_report.pdf`.

**Today at a glance**

1. Load the tagged families and check missing values and duplicates.
2. Choose the working set (complete priority years, no GaN / quantum cascade).
3. Count families per year and per company.
4. Show the technology mix and the company x technology heatmaps.
5. Look at the six tracked companies and the top keywords.
6. Write 3 to 5 observations and build the report PDF.

Run the cells from top to bottom (Run All). Each chart is followed by its data table, so every number can be checked.

In [ ]:
import re
import textwrap
from collections import Counter
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# Colours: series colours in a fixed order, one blue ramp for counts, quiet grid and axes
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
BLUE_RAMP = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"])
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linestyle": "-", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "legend.frameon": False, "figure.dpi": 110,
})

## 1. Load and check the data

One patent document is one row in `patents_tagged.csv`. One patent family (one invention, possibly filed in several countries) is one row in `families_tagged.csv`. We count families, so the same invention is not counted twice.

In [ ]:
families = pd.read_csv(ROOT / "data" / "processed" / "families_tagged.csv")
patents = pd.read_csv(ROOT / "data" / "processed" / "patents_tagged.csv")
print(f"{len(patents)} patent documents -> {len(families)} families")
print("Duplicate patent numbers:", patents["number"].duplicated().sum())
print("Duplicate families in the family table:", families["family_key"].duplicated().sum())
print("\nEmpty cells per column (families):")
print(families.isna().sum()[lambda s: s > 0].to_string())
families.head(3)

## 2. Choose the working set

- **Priority years 2010 to 2024.** The download covers publication years 2010 to 2026, so priority years before 2010 are only partly there. Patents are published about 18 months after filing, so 2025 and 2026 are still incomplete.
- **Without GaN / quantum cascade patents.** They matched the CPC set but are not InP or GaAs quantum-dot lasers. They stay in the files, flagged, and are only left out of the charts.

In [ ]:
FIRST_YEAR, LAST_YEAR = 2010, 2024
PERIODS = [(2010, 2012), (2013, 2015), (2016, 2018), (2019, 2021), (2022, 2024)]
LABELS = [f"{a}-{str(b)[2:]}" for a, b in PERIODS]
TAG_COLUMNS = ["material", "emission", "cavity", "active_region", "integration", "trend", "application"]

families["priority_year"] = pd.to_numeric(families["priority_year"], errors="coerce")
families["lead_applicant_clean"] = families["lead_applicant_clean"].fillna("")
families["applicants"] = families["applicants"].fillna("")
noise = families["noise_flag"].astype(str).eq("True")
in_range = families["priority_year"].between(FIRST_YEAR, LAST_YEAR)

core = families[in_range & ~noise].copy()
core["priority_year"] = core["priority_year"].astype(int)
core["period"] = core["priority_year"].map(
    lambda y: next(label for (a, b), label in zip(PERIODS, LABELS) if a <= y <= b)
)
print(f"Working set: {len(core)} of {len(families)} families")
print(f"  outside priority years {FIRST_YEAR}-{LAST_YEAR}: {(~in_range).sum()}")
print(f"  flagged GaN / quantum cascade: {int(noise.sum())}")


def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def tag_table(frame, column):
    """One row per (family, tag) for one tag column, without 'unspecified'."""
    table = frame[["family_key", "priority_year", "lead_applicant_clean", "period", column]].copy()
    table[column] = table[column].str.split("; ")
    table = table.explode(column).rename(columns={column: "tag"})
    return table[table["tag"] != "unspecified"]


def heatmap(table, title, name, figsize):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(table.values, cmap=BLUE_RAMP, aspect="auto", vmin=0)
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=30, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    top = max(table.values.max(), 1)
    for (i, j), value in np.ndenumerate(table.values):
        if value:
            ax.text(j, i, int(value), ha="center", va="center", fontsize=8,
                    color="white" if value > 0.55 * top else INK)
    ax.set_title(title)
    save(fig, name)

## 3. Patent families per year

Priority year is the year of the first filing, the closest date to the invention. The tagged lines (InP, GaAs QD) only count families whose title, abstract or CPC codes name the material, so they are lower bounds.

In [ ]:
years = range(FIRST_YEAR, LAST_YEAR + 1)
per_year = core.groupby("priority_year").size().reindex(years, fill_value=0)
material = tag_table(core, "material")
by_material = material.groupby(["priority_year", "tag"]).size().unstack(fill_value=0).reindex(years, fill_value=0)

print(pd.DataFrame({"All families": per_year}).join(by_material).to_string())

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(per_year.index, per_year.values, color=SERIES[0], lw=2, marker="o", ms=4, label="All families")
for color, tag in zip(SERIES[1:], by_material.columns):
    ax.plot(by_material.index, by_material[tag], color=color, lw=2, marker="o", ms=4, label=f"{tag} (tagged)")
ax.set_title("Patent families per priority year")
ax.set_xlabel("Priority year")
ax.set_ylabel("Families")
ax.legend(loc="upper left")
save(fig, "01_families_per_year")

## 4. Who owns the patents

The lead applicant is the first company named on the patent. Company names are only lightly cleaned here (suffixes such as Inc, Ltd, GmbH and Group are removed). Full name merging is a Week 2 task.

In [ ]:
named = core[core["lead_applicant_clean"] != ""]
top = named["lead_applicant_clean"].value_counts().head(15)
print(top.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index[::-1], top.values[::-1], color=SERIES[0], height=0.6)
ax.set_title("Top 15 companies by patent families (lead applicant)")
ax.set_xlabel("Families")
ax.grid(axis="y", visible=False)
save(fig, "02_top_companies")

In [ ]:
top12 = named["lead_applicant_clean"].value_counts().head(12).index
momentum = (
    named[named["lead_applicant_clean"].isin(top12)]
    .groupby(["lead_applicant_clean", "period"]).size().unstack(fill_value=0)
    .reindex(index=top12, columns=LABELS, fill_value=0)
)
momentum["change"] = momentum[LABELS[-1]] - momentum[LABELS[2]]
print(f"change = families in {LABELS[-1]} minus families in {LABELS[2]}")
print(momentum.sort_values("change", ascending=False).to_string())
heatmap(momentum[LABELS], "Families per period, top 12 companies", "03_company_momentum", (7, 4.6))

## 5. Technology mix over time

Stacked bars by three-year period. A family can carry more than one tag in a dimension, so the bars count tags, not families. Most families carry no tag in these dimensions, so read the shape, not the total height.

In [ ]:
def stacked_by_period(column, title, name):
    table = tag_table(core, column)
    counts = table.groupby(["period", "tag"]).size().unstack(fill_value=0).reindex(LABELS, fill_value=0)
    if counts.empty:
        print(f"No tagged families for {column}")
        return
    counts = counts[counts.sum().sort_values(ascending=False).index]
    print(title)
    print(counts.to_string())
    fig, ax = plt.subplots(figsize=(7, 3.8))
    bottom = np.zeros(len(counts))
    for color, tag in zip(SERIES, counts.columns):
        ax.bar(counts.index, counts[tag], bottom=bottom, color=color, width=0.6,
               edgecolor=SURFACE, linewidth=1.2, label=tag)
        bottom += counts[tag].values
    ax.set_title(title)
    ax.set_xlabel("Priority-year period")
    ax.set_ylabel("Tags (a family can carry several)")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left", fontsize=8)
    save(fig, name)


stacked_by_period("cavity", "Cavity type over time", "04_cavity_mix")
stacked_by_period("emission", "Emission type over time", "05_emission_mix")
stacked_by_period("integration", "Integration over time", "06_integration_mix")

## 6. Company specialisation and possible white space

The first heatmap shows which companies carry which tags. The second crosses material with application and integration: empty or thin cells are candidate gaps, but check them before believing them. Most families have no application tag, and the tags come from titles, abstracts and CPC codes only, not from claims.

In [ ]:
long = pd.concat([tag_table(core, column).assign(dimension=column) for column in TAG_COLUMNS])
tag_order = long["tag"].value_counts().head(14).index
company_tag = (
    long[long["lead_applicant_clean"].isin(top12)]
    .groupby(["lead_applicant_clean", "tag"]).size().unstack(fill_value=0)
    .reindex(index=top12, columns=tag_order, fill_value=0)
)
print(company_tag.to_string())
heatmap(company_tag, "Company x technology tag (families)", "07_company_by_tag", (9, 4.8))

In [ ]:
MATERIALS = ["InP", "GaAs QD"]
APPLICATIONS = ["Datacenter", "Telecom", "Sensing / LiDAR"]
INTEGRATION = ["Photonic IC", "Heterogeneous on Si/SiN", "Grown on Si"]


def cross(rows, cols, row_order, col_order):
    left = tag_table(core, rows)[["family_key", "tag"]].rename(columns={"tag": rows})
    right = tag_table(core, cols)[["family_key", "tag"]].rename(columns={"tag": cols})
    merged = left.merge(right, on="family_key")
    return pd.crosstab(merged[rows], merged[cols]).reindex(index=row_order, columns=col_order, fill_value=0)


material_application = cross("material", "application", MATERIALS, APPLICATIONS)
material_integration = cross("material", "integration", MATERIALS, INTEGRATION)
print(material_application.to_string(), "\n")
print(material_integration.to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
for ax, table, title in zip(axes, [material_application, material_integration],
                            ["Material x application", "Material x integration"]):
    ax.imshow(table.values, cmap=BLUE_RAMP, aspect="auto", vmin=0)
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=20, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    top_value = max(table.values.max(), 1)
    for (i, j), value in np.ndenumerate(table.values):
        ax.text(j, i, int(value), ha="center", va="center", fontsize=9,
                color="white" if value > 0.55 * top_value else INK)
    ax.set_title(title)
fig.tight_layout()
save(fig, "08_white_space_cells")

## 7. Is the field growing or consolidating?

Families and different lead applicants per year. Both rising means a field that is still attracting new players. Filings rising while the number of companies falls points to consolidation.

In [ ]:
life = core.groupby("priority_year").agg(
    families=("family_key", "nunique"),
    companies=("lead_applicant_clean", lambda s: s[s != ""].nunique()),
).reindex(years, fill_value=0)
print(life.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(life.index, life["families"], color=SERIES[0], lw=2, marker="o", ms=4, label="Families")
ax.plot(life.index, life["companies"], color=SERIES[1], lw=2, marker="o", ms=4, label="Different lead applicants")
ax.set_title("Filings and applicants per priority year")
ax.set_xlabel("Priority year")
ax.set_ylabel("Count")
ax.legend(loc="upper left")
save(fig, "09_life_cycle")

## 8. The six tracked companies

Matched on any applicant name, with a few known name variants. This is a first approximation. Proper name merging (subsidiaries, takeovers) comes in Week 2.

In [ ]:
TRACKED = {
    "Coherent (incl. II-VI, Finisar)": r"COHERENT|II-VI|II VI|FINISAR",
    "Lumentum (incl. Oclaro, JDSU)": r"LUMENTUM|OCLARO|JDS UNIPHASE",
    "ams OSRAM": r"OSRAM|\bAMS\b",
    "Trumpf": r"TRUMPF",
    "Broadcom (incl. Avago)": r"BROADCOM|AVAGO",
    "Nokia Bell Labs (incl. Alcatel)": r"NOKIA|ALCATEL|BELL LAB",
}
names_upper = core["applicants"].str.upper()
rows = {}
for company, pattern in TRACKED.items():
    hit = core[names_upper.str.contains(pattern, regex=True)]
    rows[company] = hit.groupby("period").size().reindex(LABELS, fill_value=0)
tracked = pd.DataFrame(rows).T
tracked["total"] = tracked.sum(axis=1)
print(tracked.to_string())
heatmap(tracked[LABELS], "Tracked companies: families per period", "10_tracked_companies", (7, 3.4))

## 9. Top keywords

For each word, the number of families whose title or abstract contains it. Very common patent words are removed.

In [ ]:
STOP = set("""the and for with said that which from into this has are can also wherein comprising comprises
including includes included provided provides present invention disclosure method device apparatus according
least one two first second using used use based obtained capable configured having have been being thereof
thereby therein such more than other each between through when where while may not its their these those
there then them they will does the""".split())


def words(text):
    return [w for w in re.findall(r"[a-z][a-z\-]{2,}", text.lower()) if w not in STOP]


word_counter, pair_counter = Counter(), Counter()
for title, abstract in zip(core["title"].fillna(""), core["abstract"].fillna("")):
    tokens = words(title + " " + abstract)
    word_counter.update(set(tokens))
    pair_counter.update(set(zip(tokens, tokens[1:])))

top_words = pd.Series(word_counter).sort_values(ascending=False).head(20)
top_pairs = pd.Series({" ".join(p): n for p, n in pair_counter.most_common(20)})
print("Top words:\n", top_words.to_string(), "\n\nTop word pairs:\n", top_pairs.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_words.index[::-1], top_words.values[::-1], color=SERIES[0], height=0.6)
ax.set_title("Top 20 words in titles and abstracts (families)")
ax.set_xlabel("Families containing the word")
ax.grid(axis="y", visible=False)
save(fig, "11_top_words")

## 10. Your observations

Look at the tables and charts above and write 3 to 5 sentences, each with a number. Good observations say what changed, by how much, and who is behind it. Replace the examples below, then run the next cell to build the report.

In [ ]:
OBSERVATIONS = [
    "Observation 1: replace this with a finding and its number, for example the growth of families per year.",
    "Observation 2: replace this with a finding about companies.",
    "Observation 3: replace this with a finding about the technology mix.",
]

In [ ]:
def text_page(pdf, title, paragraphs):
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.08, 0.95, title, fontsize=16, fontweight="bold", va="top", color=INK)
    y = 0.90
    for paragraph in paragraphs:
        wrapped = textwrap.fill(paragraph, 98)
        fig.text(0.08, y, wrapped, fontsize=9.5, va="top", color=INK2)
        y -= 0.0165 * (wrapped.count("\n") + 1) + 0.014
    pdf.savefig(fig)
    plt.close(fig)


share = lambda column: 100 * (core[column] != "unspecified").mean()
summary = [
    f"Dataset: {len(patents)} patent documents from the EPO Open Patent Services (publication years 2010 to 2026), "
    f"grouped into {len(families)} patent families.",
    f"Working set for the charts: {len(core)} families with priority years {FIRST_YEAR} to {LAST_YEAR}, "
    f"without {int(noise.sum())} families flagged as GaN or quantum cascade lasers.",
    "Top companies by families (lead applicant): "
    + ", ".join(f"{name} ({count})" for name, count in top.head(5).items()) + ".",
    f"Tag coverage: {share('material'):.0f}% of families carry a material tag, {share('emission'):.0f}% an emission tag, "
    f"{share('application'):.0f}% an application tag.",
    "Limits: tags come from keyword rules on titles, abstracts and CPC codes, not from claims. Years before 2010 are not "
    "collected. Company names are only lightly cleaned. Counts follow the EPO's one-result-per-application grouping. "
    "The latest priority years are incomplete because patents are published about 18 months after filing.",
]

with PdfPages(ROOT / "reports" / "week1_report.pdf") as pdf:
    text_page(pdf, "AI Patent Intelligence: Week 1 report", summary)
    for path in sorted(FIGURES.glob("[01][0-9]_*.png")):
        fig = plt.figure(figsize=(8.27, 11.69))
        ax = fig.add_axes([0.06, 0.08, 0.88, 0.84])
        ax.imshow(plt.imread(path))
        ax.set_anchor("N")
        ax.axis("off")
        pdf.savefig(fig)
        plt.close(fig)
    text_page(pdf, "Initial observations", OBSERVATIONS)
print("Saved", ROOT / "reports" / "week1_report.pdf")